# H&M 고객 데이터 문제

In [1]:
import pandas as pd

df = pd.read_csv("customer_hm.csv")
print(df.shape)
df.head()

(1048575, 6)


,customer_id,FN,Active,club_member_status,fashion_news_frequency,age
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0,0,ACTIVE,NONE,49
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,0,0,ACTIVE,NONE,25
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0,0,ACTIVE,NONE,24
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,0,0,ACTIVE,NONE,54
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,1,1,ACTIVE,Regularly,52


---

### 문제 1 — 그룹화: `club_member_status`별 고객 수
• `club_member_status`로 그룹화해서 **고객 수**를 구하세요. (`groupby` + `size` 또는 `value_counts`)  
• 결과는 내림차순으로 정렬해봅시다.

In [ ]:
# TODO: 여기에서 풀이 코드를 작성하세요.
df.groupby('club_member_status').size()

club_member_status
ACTIVE        982635
LEFT CLUB        359
PRE-CREATE     65581
Name: customer_id, dtype: int64

---

### 문제 2 — 집계표(agg): 상태별 고객 수 + 비율(%)
• 1번 결과를 조금 더 보고서형으로 바꿔봅시다.  
• `club_member_status`별 `count`, `ratio(%)`가 있는 DataFrame을 만드세요.

In [9]:
# TODO: 여기에서 풀이 코드를 작성하세요.
status_summary = df.groupby('club_member_status').agg(count=("customer_id", "size")).sort_values("count", ascending=False)
status_summary['ratio_pct'] = status_summary['count'] / status_summary['count'].sum() * 100
status_summary
# assign

,count,ratio_pct
club_member_status,,
ACTIVE,982635,93.711466
PRE-CREATE,65581,6.254297
LEFT CLUB,359,0.034237


---

### 문제 3 — 그룹화 + 다중 집계: `fashion_news_frequency`별 나이 요약
• `fashion_news_frequency`별로 **고객 수**, **평균 나이**, **중앙값 나이**를 한 번에 구하세요. (`groupby` + `agg`)  
• 평균 나이는 소수점 2자리로 반올림해봅시다.  
• `count`가 내림차순으로 정렬하세요

In [14]:
# TODO: 여기에서 풀이 코드를 작성하세요.
age_summary = df.groupby('fashion_news_frequency').agg(
    count = ('customer_id', 'size'),
    age_mean = ('age', 'mean'),
    age_median = ('age', 'median')
).assign(age_mean=lambda x: x['age_mean'].round(2)).sort_values("count", ascending=False)
age_summary

,count,age_mean,age_median
fashion_news_frequency,,,
NONE,674698,36.00,31.0
Regularly,373218,37.03,32.0
Monthly,658,39.39,36.5


---

### 문제 4 — 복수 기준 그룹화: `club_member_status × Active` 요약
• `club_member_status`와 `Active`로 묶어서 다음을 구하세요.  
  1) 고객 수(`size`)  
  2) 평균 나이(`mean`)  
  3) `FN` 평균(=비율로 해석 가능)  

In [15]:
# TODO: 여기에서 풀이 코드를 작성하세요.
df.groupby(['club_member_status', 'Active']).agg(
    count = ('customer_id', 'size'),
    age_mean = ('age', 'mean'),
    FN_mean = ('FN', 'mean')
)

count   age_mean   FN_mean
club_member_status Active                             
ACTIVE             0       624068  35.537385  0.015623
                   1       358567  37.003241  1.000000
LEFT CLUB          0          356  33.750000  0.002809
                   1            3  44.333333  1.000000
PRE-CREATE         0        61205  40.458296  0.001846
                   1         4376  46.056673  1.000000

---

### 문제 5 — transform: 상태별 평균 나이를 원본 길이 유지로 붙이기
• `club_member_status`별 평균 나이를 각 행에 붙여 `age_mean_by_status` 컬럼을 만드세요. (`transform`)  
• 그리고 `age_diff` = `age - age_mean_by_status` 컬럼도 추가하세요.  
• `[['club_member_status','age','age_mean_by_status','age_diff']]` 5행만 확인!

In [ ]:
# TODO: 여기에서 풀이 코드를 작성하세요.
df['age_mean_by_status'] = df.groupby('club_member_status')['age'].transform("mean")
df['age_diff'] = df['age'] - df['age_mean_by_status']

df[['club_member_status','age','age_mean_by_status','age_diff']].head()

,club_member_status,age,age_mean_by_status,age_diff
0,ACTIVE,49,36.072281,12.927719
1,ACTIVE,25,36.072281,-11.072281
2,ACTIVE,24,36.072281,-12.072281
3,ACTIVE,54,36.072281,17.927719
4,ACTIVE,52,36.072281,15.927719


---

### 문제 6 — transform(응용): (상태, 뉴스빈도) 그룹 크기 붙이기
• `club_member_status`와 `fashion_news_frequency` 조합별로 그룹 크기(고객 수)를 구해서  
  각 행에 `group_size`로 붙이세요. (`transform('size')`)  
• `group_size`가 큰 조합 TOP 5를 확인해봅시다. (힌트: `drop_duplicates` 활용)

In [31]:
# TODO: 여기에서 풀이 코드를 작성하세요.
df['group_size'] = df.groupby(['club_member_status', 'fashion_news_frequency'])['customer_id'].transform("size")
df[['group_size']].drop_duplicates().sort_values("group_size", ascending = False).head()


,group_size
0,613304.0
4,368719.0
33,61039.0
24,4495.0
4911,611.0


---

### 문제 7 — unstack: 상태 × 뉴스빈도 고객 수 표 만들기
• `club_member_status × fashion_news_frequency` 고객 수를 구한 뒤, 표 형태로 펼치세요. (`unstack`)  
• 비어있는 조합은 0으로 채우세요. (`fill_value=0`)

In [34]:
# TODO: 여기에서 풀이 코드를 작성하세요.
df.groupby(['club_member_status', 'fashion_news_frequency'])['customer_id'].size().unstack(fill_value=0)

fashion_news_frequency,Monthly,NONE,Regularly
club_member_status,,,
ACTIVE,611,613304,368719
LEFT CLUB,0,355,4
PRE-CREATE,47,61039,4495


---

### 문제 8 — pivot_table: 상태 × 뉴스빈도 활동률(Active 평균)
• `pivot_table`로 아래 표를 만드세요.  
  - index: `club_member_status`  
  - columns: `fashion_news_frequency`  
  - values: `Active` (평균 = 활동률)  
  - 빈 값은 0으로 채우기

In [ ]:
# TODO: 여기에서 풀이 코드를 작성하세요.
df.pivot_table(
    index = 'club_member_status',
    columns = 'fashion_news_frequency',
    values="Active",
    aggfunc = "mean",
    fill_value = 0
)

fashion_news_frequency,Monthly,NONE,Regularly
club_member_status,,,
ACTIVE,0.944354,0.000623,0.969866
LEFT CLUB,0.000000,0.000000,0.750000
PRE-CREATE,0.978723,0.000082,0.962180


---

### 문제 9 — melt: 8번 피벗 표를 long(그래프용)으로 바꾸기
• 8번 `active_pivot`을 long 형태로 바꾸세요.  
• 결과 컬럼은 `club_member_status`, `fashion_news_frequency`, `active_rate`가 되도록!

In [ ]:
# TODO: 여기에서 풀이 코드를 작성하세요.
active_pivot.reset_index().melt(
    id_vars = 
)

---

### 문제 10 — pivot: 9번 long 데이터를 다시 wide로 복원
• 9번 `active_long`을 `pivot`으로 다시 wide로 돌려보세요.  
• (응용-검색해보세요(`equals`??))그리고 원래의 `active_pivot`과 같은지 확인해봅시다. (힌트: `equals`)

In [ ]:
# TODO: 여기에서 풀이 코드를 작성하세요.


---

### 문제 11 — 문자열 정리: `fashion_news_frequency` 클린 버전 만들기
• `fashion_news_frequency`를 아래 규칙으로 정리해 `fn_clean` 컬럼을 만드세요.  
  - 결측치 -> `'unknown'`  
  - 양끝 공백 제거  
  - 소문자화  
• 정리 전/후 `value_counts()` 비교까지! -> `dropna=False` 옵션을 주자(그래야 결측 보임)

In [ ]:
# TODO: 여기에서 풀이 코드를 작성하세요.


---

### 문제 12 — 문자열: `customer_id` prefix(접두사)/suffix(접미사) 추출
• `customer_id`에서  
  - 앞 2글자: `id_prefix2`  
  - 뒤 3글자: `id_suffix3`  
  컬럼을 만들고, `id_prefix2` 상위 10개 빈도를 출력하세요.

In [ ]:
# TODO: 여기에서 풀이 코드를 작성하세요.


---

### 문제 13 — map: `club_member_status` 한글 라벨 붙이기
• 아래 매핑을 사용해 `status_ko` 컬럼을 만드세요.  
  - `ACTIVE` -> `활동중`  
  - `PRE-CREATE` -> `사전생성`  
  - `LEFT CLUB` -> `탈퇴`  
• 그리고 `status_ko`별 고객 수를 출력하세요.

In [ ]:
# TODO: 여기에서 풀이 코드를 작성하세요.


---

### 문제 14 — apply/lambda: 나이대(`age_band`) 구간화 + 활동률 보기
• 나이를 아래 구간으로 나눠 `age_band`를 만드세요. (apply 또는 함수 + apply)  
  - 16~19: `10대`  
  - 20~29: `20대`  
  - 30~39: `30대`  
  - 40~59: `40~50대`  
  - 60+: `60대+`  
• `age_band`별 고객 수와 활동률(`Active` 평균)을 함께 구하세요.  
• 행(인덱스)의 순서를 보장하기 위한 `sort_index`를 해주는건 좋은 방법!

In [ ]:
# TODO: 여기에서 풀이 코드를 작성하세요.


---

### 문제 15 — set_index / loc: 고객 ID로 빠르게 조회하기
• `customer_id`를 인덱스로 설정한 `df_idx`를 만드세요.  
• 그리고 아래처럼 **임의 5명**을 뽑아(`sample`) 해당 고객의 `age`, `status_ko`, `fn_clean`을 조회해보세요.  
  - 힌트: `ids = df['customer_id'].sample(5, random_state=42).tolist()`  
• 마지막에 `reset_index()`로 원복까지!

In [ ]:
# TODO: 여기에서 풀이 코드를 작성하세요.


---

### 문제 16 — reset_index(정리): 표 결과를 컬럼형으로 만들기
• 7번에서 만든 `status_fn_table`은 인덱스가 `club_member_status`입니다.  
• 이를 `reset_index()`로 풀어서 `club_member_status`가 컬럼으로 보이게 만들어보세요.

In [ ]:
# TODO: 여기에서 풀이 코드를 작성하세요.


---

### 문제 17 — merge: 뉴스 빈도 점수표 붙이기
• 아래 조건의 매핑 테이블을 DataFrame으로 만들고, `fn_clean` 기준으로 `left merge` 하세요.  
  - `none: 0, monthly: 1, regularly: 2, unknown: -1`  
• 병합 후 `fn_score`별 평균 나이를 확인해봅시다.

In [ ]:
# TODO: 여기에서 풀이 코드를 작성하세요.


---

### 문제 18 — concat + 최종 reshape: `age_band × status_ko` 활동률 표 만들기
• 1. `Active==1`과 `Active==0`으로 나눴다가 다시 `concat`으로 합쳐보세요.  
  - 원본 행 수가 같은지 체크!  

• 2. 그 다음, `pivot_table`로 `age_band(행) × status_ko(열)`의 **활동률(Active 평균)** 표를 만들고  
  `melt`로 long 형태까지 만들어보세요.

In [ ]:
# TODO: 여기에서 풀이 코드를 작성하세요.
